# nb_editing_tools

> MCP tools for reading, editing, and analyzing notebook cells

In [ ]:
#| default_exp tools.editing

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

import re
import json
from pathlib import Path

from rich.panel import Panel
from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, 
)
from nbdev_mcp.utils.nb import join_source, truncate
from nbdev_mcp.utils.md import split_markdown_into_cells
from nbdev_mcp.utils.rich import make_console, get_output, render_table

## Check Generated Files

In [ ]:
#| export
def check_if_generated(
    project: Optional[str] = None,
    file_path: str = ''
) -> Dict[str, Any]:
    """Check if a file is auto-generated by nbdev.
    
    USE THIS BEFORE EDITING ANY PYTHON FILE IN A NBDEV PROJECT!
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    file_path : str
        Path to the file to check.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'is_generated' bool and suggested action.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib_name = s.get('lib_name', '')
    is_in_lib = lib_name and (file_path.startswith(lib_name + '/') or file_path.startswith(lib_name + '\\'))
    is_py_file = file_path.endswith('.py')
    is_readme = file_path == 'README.md' or file_path.endswith('/README.md')
    
    if is_in_lib and is_py_file:
        nbs = nbs_dir(p)
        return {
            'ok': True,
            'is_generated': True,
            'file': file_path,
            'warning': '⚠️ This file is GENERATED by nbdev. DO NOT EDIT DIRECTLY!',
            'action': f"1. Use find_source_notebook(py_file='{file_path}') to find the source notebook\n2. Edit that notebook instead\n3. Run nbdev_export() to regenerate this file",
            'pretty': f'⚠️ {file_path} is AUTO-GENERATED\nEdit the source notebook in {nbs}/ instead!'
        }
    
    if is_readme:
        return {
            'ok': True,
            'is_generated': True,
            'file': file_path,
            'warning': '⚠️ README.md is GENERATED from index.ipynb. DO NOT EDIT DIRECTLY!',
            'action': 'Edit nbs/index.ipynb instead, then run nbdev_readme()',
            'pretty': '⚠️ README.md is AUTO-GENERATED\nEdit nbs/index.ipynb instead!'
        }
    
    return {
        'ok': True,
        'is_generated': False,
        'file': file_path,
        'message': '✓ This file is safe to edit (not generated by nbdev)'
    }

In [ ]:
#| export
def find_source_notebook(
    project: Optional[str] = None,
    py_file: str = ''
) -> Dict[str, Any]:
    """Find which notebook generated a given Python file.
    
    CRITICAL: Use this to find the source before editing any .py file!
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    py_file : str
        Path to the Python file.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'notebook' path if found.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    if not py_file:
        return {'ok': False, 'error': 'py_file parameter is required'}
    
    s = settings_dict(p)
    lib_name = s.get('lib_name', '')
    nbs = nbs_dir(p)
    py_path = Path(py_file)
    
    if lib_name and py_file.startswith(lib_name):
        module_name = py_file.replace(lib_name + '/', '').replace(lib_name + '\\', '')
        module_name = Path(module_name).stem
    else:
        module_name = py_path.stem
    
    matching_notebooks = []
    if nbs.exists():
        for nb_path in nbs.rglob('*.ipynb'):
            if '.ipynb_checkpoints' in str(nb_path):
                continue
            try:
                nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
                for cell in nb_data.get('cells', []):
                    if cell.get('cell_type') == 'code':
                        source = join_source(cell.get('source', []))
                        match = re.search(r'#\|\s*default_exp\s+(\w+)', source)
                        if match and match.group(1) == module_name:
                            matching_notebooks.append({
                                'notebook': str(nb_path.relative_to(p)),
                                'full_path': str(nb_path),
                                'module': module_name,
                                'confidence': 'high'
                            })
                            break
                        if nb_path.stem == module_name:
                            matching_notebooks.append({
                                'notebook': str(nb_path.relative_to(p)),
                                'full_path': str(nb_path),
                                'module': module_name,
                                'confidence': 'medium'
                            })
                            break
            except Exception:
                continue
    
    if not matching_notebooks:
        return {
            'ok': False,
            'error': f'Could not find source notebook for {py_file}',
            'hint': f"Expected notebook with '#| default_exp {module_name}' in {nbs}/"
        }
    
    best_match = sorted(matching_notebooks, key=lambda x: x['confidence'], reverse=True)[0]
    c = make_console()
    c.print(Panel(
        f"[bold green]✓[/] {py_file} is generated from:\n[cyan]{best_match['notebook']}[/]",
        title='Source Notebook Found'
    ))
    
    return {
        'ok': True,
        'py_file': py_file,
        'notebook': best_match['notebook'],
        'notebook_full_path': best_match['full_path'],
        'module': module_name,
        'message': f"✓ Source: {best_match['notebook']}",
        'pretty': get_output(c)
    }

## Analyze Exports

In [ ]:
#| export
def analyze_exports(
    project: Optional[str] = None,
    notebook: str = '',
    preview_length: int = 200
) -> Dict[str, Any]:
    """Analyze what a notebook exports (functions, classes, etc.).
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename relative to nbs/ directory.
    preview_length : int
        Maximum chars for cell preview (default 200).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'exports' list of exported symbols.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    exports = []
    module_name = None
    
    for i, cell in enumerate(nb_data.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue
        source = join_source(cell.get('source', []))
        
        match = re.search(r'#\|\s*default_exp\s+(\w+)', source)
        if match:
            module_name = match.group(1)
        
        has_export_directive = bool(re.search(r'#\|\s*export\s*$', source, re.MULTILINE))
        has_exporti = bool(re.search(r'#\|\s*exporti', source))
        
        if has_export_directive or has_exporti:
            symbols = []
            for func_match in re.finditer(r'def\s+(\w+)\s*\(', source):
                symbols.append({'type': 'function', 'name': func_match.group(1)})
            for class_match in re.finditer(r'class\s+(\w+)\s*[\(:]', source):
                symbols.append({'type': 'class', 'name': class_match.group(1)})
            
            exports.append({
                'cell_index': i,
                'export_type': 'exporti' if has_exporti else 'export',
                'symbols': symbols,
                'preview': truncate(source, preview_length)
            })
    
    rows = []
    for exp in exports[:50]:
        symbols_str = ', '.join(s['name'] for s in exp['symbols']) or '(no named symbols)'
        rows.append([str(exp['cell_index']), exp['export_type'], symbols_str])
    if len(exports) > 50:
        rows.append(['...', '...', f'+ {len(exports) - 50} more'])
    pretty = render_table(f'Exports from {notebook}', ['Cell', 'Type', 'Symbols'], rows)
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'module': module_name,
        'export_count': len(exports),
        'exports': exports[:50],
        'total_exports': len(exports),
        'limited': len(exports) > 50,
        'pretty': pretty
    }

## Cell Reading and Editing

In [ ]:
#| export
def read_notebook_cell(
    project: Optional[str] = None,
    notebook: str = '',
    cell_index: Optional[int] = None,
    search: Optional[str] = None,
    max_results: int = 10,
    truncate_length: int = 2000
) -> Dict[str, Any]:
    """Read specific cells from a notebook by index or search.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename.
    cell_index : int, optional
        Specific cell index to read.
    search : str, optional
        Search string to find in cells.
    max_results : int
        Maximum search results (default 10).
    truncate_length : int
        Max chars per cell (default 2000).
    
    Returns
    -------
    Dict[str, Any]
        Result with cell content or search results.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    cells = nb_data.get('cells', [])
    
    if cell_index is not None:
        if cell_index < 0 or cell_index >= len(cells):
            return {'ok': False, 'error': f'Cell index {cell_index} out of range (0-{len(cells)-1})'}
        cell = cells[cell_index]
        source = join_source(cell.get('source', []))
        return {
            'ok': True,
            'notebook': str(nb_path.relative_to(p)),
            'cell_index': cell_index,
            'cell_type': cell.get('cell_type'),
            'source': truncate(source, truncate_length),
            'source_length': len(source),
            'truncated': len(source) > truncate_length
        }
    
    if search:
        matching_cells = []
        for i, cell in enumerate(cells):
            if len(matching_cells) >= max_results:
                break
            source = join_source(cell.get('source', []))
            if search.lower() in source.lower():
                matching_cells.append({
                    'cell_index': i,
                    'cell_type': cell.get('cell_type'),
                    'source': truncate(source, truncate_length),
                    'source_length': len(source),
                    'truncated': len(source) > truncate_length
                })
        
        total_matches = sum(
            1 for cell in cells
            if search.lower() in join_source(cell.get('source', [])).lower()
        )
        
        return {
            'ok': True,
            'notebook': str(nb_path.relative_to(p)),
            'search': search,
            'total_matches': total_matches,
            'returned': len(matching_cells),
            'limited': total_matches > max_results,
            'cells': matching_cells,
            'message': f'Showing {len(matching_cells)} of {total_matches} matches' if total_matches > max_results else f'Found {total_matches} matches'
        }
    
    return {'ok': False, 'error': 'Either cell_index or search must be provided'}

In [ ]:
#| export
def edit_notebook_cell(
    project: Optional[str] = None,
    notebook: str = '',
    cell_index: int = 0,
    new_source: str = ''
) -> Dict[str, Any]:
    """Edit a specific cell in a notebook.
    
    Run nbdev_export afterward to regenerate Python modules!
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename.
    cell_index : int
        Index of cell to edit.
    new_source : str
        New source code for the cell.
    
    Returns
    -------
    Dict[str, Any]
        Result with confirmation and next steps.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    cells = nb_data.get('cells', [])
    if cell_index < 0 or cell_index >= len(cells):
        return {'ok': False, 'error': f'Cell index {cell_index} out of range (0-{len(cells)-1})'}
    
    cells[cell_index]['source'] = new_source.splitlines(True)
    
    try:
        nb_path.write_text(json.dumps(nb_data, indent=2), encoding='utf-8')
    except Exception as e:
        return {'ok': False, 'error': f'Could not write notebook: {e}'}
    
    c = make_console()
    c.print(Panel(
        f'[green]✓[/] Updated cell {cell_index} in {notebook}\n\n[yellow]⚠ Next step: Run nbdev_export()[/]',
        title='Cell Edited'
    ))
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'cell_index': cell_index,
        'message': f'✓ Cell {cell_index} updated',
        'next_step': 'Run nbdev_export() to regenerate Python modules',
        'pretty': get_output(c)
    }

In [ ]:
#| export
def add_notebook_cell(
    project: Optional[str] = None,
    notebook: str = '',
    source: str = '',
    cell_type: str = 'code',
    position: str = 'end',
    after_index: Optional[int] = None
) -> Dict[str, Any]:
    """Add a new cell to a notebook.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    notebook : str
        Notebook filename.
    source : str
        Source code for the new cell.
    cell_type : str
        Cell type ('code' or 'markdown').
    position : str
        Where to insert: 'start', 'end', or 'after'.
    after_index : int, optional
        Cell index to insert after (required if position='after').
    
    Returns
    -------
    Dict[str, Any]
        Result with inserted cell index.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    nb_path = (nbs / notebook).resolve() if '/' not in notebook else (p / notebook).resolve()
    
    if not nb_path.exists():
        return {'ok': False, 'error': f'Notebook not found: {notebook}'}
    
    try:
        nb_data = json.loads(nb_path.read_text(encoding='utf-8'))
    except Exception as e:
        return {'ok': False, 'error': f'Could not read notebook: {e}'}
    
    new_cell = {
        'cell_type': cell_type,
        'metadata': {},
        'source': source.splitlines(True)
    }
    if cell_type == 'code':
        new_cell['execution_count'] = None
        new_cell['outputs'] = []
    
    cells = nb_data.get('cells', [])
    
    if position == 'start':
        cells.insert(0, new_cell)
        insert_idx = 0
    elif position == 'end':
        cells.append(new_cell)
        insert_idx = len(cells) - 1
    elif position == 'after' and after_index is not None:
        if after_index < 0 or after_index >= len(cells):
            return {'ok': False, 'error': f'after_index {after_index} out of range'}
        cells.insert(after_index + 1, new_cell)
        insert_idx = after_index + 1
    else:
        return {'ok': False, 'error': 'Invalid position or missing after_index'}
    
    nb_data['cells'] = cells
    
    try:
        nb_path.write_text(json.dumps(nb_data, indent=2), encoding='utf-8')
    except Exception as e:
        return {'ok': False, 'error': f'Could not write notebook: {e}'}
    
    return {
        'ok': True,
        'notebook': str(nb_path.relative_to(p)),
        'inserted_at': insert_idx,
        'message': f'✓ Added {cell_type} cell at index {insert_idx}',
        'next_step': 'Run nbdev_export() if this is an export cell'
    }

In [ ]:
#| export
def split_markdown_cells(markdown: str) -> Dict[str, Any]:
    """Split a markdown blob into notebook-friendly markdown cells.
    
    Rules:
    - Each heading line (``#``, ``##``, ...) is its own cell
    - Text after a heading becomes a separate cell
    - Reference link definitions are propagated to cells that use them
    
    Parameters
    ----------
    markdown : str
        Full markdown document text.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'cells' list ready for notebook insertion.
    """
    cells = split_markdown_into_cells(markdown)
    preview = [{'index': i, 'source': c['source'][:160]} for i, c in enumerate(cells[:5])]
    
    return {
        'ok': True,
        'count': len(cells),
        'cells': cells,
        'preview': preview,
        'message': f'Split into {len(cells)} markdown cells'
    }

## MCP Registration

In [ ]:
#| export
def add_notebook_editing_tools(mcp: FastMCP) -> None:
    """Attach notebook editing tools to the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    mcp.add_tool(check_if_generated)
    mcp.add_tool(find_source_notebook)
    mcp.add_tool(analyze_exports)
    mcp.add_tool(read_notebook_cell)
    mcp.add_tool(edit_notebook_cell)
    mcp.add_tool(add_notebook_cell)
    mcp.add_tool(split_markdown_cells)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()